## Bonus: HuggingFace Skills — Cloud Fine-Tuning with Claude Code

The workflow we followed in this lab (load dataset, load model, configure training, fine-tune, evaluate) can also be done entirely through natural language using **HuggingFace Skills** with Claude Code.

**Video Tutorial:** [Fine-Tune an Open Source LLM with Claude Code (HuggingFace Model Trainer Skill)](https://www.youtube.com/watch?v=HGPTUc7tEq4)
**Blog Post:** [We Got Claude to Fine-Tune an Open Source LLM (HuggingFace Blog)](https://huggingface.co/blog/hf-skills-training)

### How It Works

1. **Install the HuggingFace Skills MCP server in Claude Code:**
```bash
claude mcp add --transport http hf-skills https://huggingface.co/mcp?bouquet=skills --header "Authorization: Bearer $HF_TOKEN"
```

2. **Instruct Claude Code in natural language:**
```
Fine-tune Qwen3-0.6B on open-r1/codeforces-cots for instruction following.
```

3. **Claude Code handles everything:** model selection, GPU allocation, training configuration, job submission, monitoring, and pushing the final model to the Hub.

### Supported Training Methods

| Method | Use Case | Example |
|--------|----------|---------|
| **SFT** (Supervised Fine-Tuning) | Input/output pairs | Customer support, Q&A, code generation |
| **DPO** (Direct Preference Optimization) | Preference alignment | Chosen vs rejected responses |
| **GRPO** (Group Relative Policy Optimization) | Verifiable tasks | Math reasoning, code correctness |

### Key Takeaway

Whether you fine-tune locally (as in this lab) or on cloud GPUs (via HuggingFace Skills), the core concepts are the same: choose a pre-trained model, prepare your dataset, configure training, and evaluate results. The "vibe coding" approach lets you do this conversationally!

## 1. Setup and Installation

In [ ]:
# Run this cell in Google Colab to install dependencies
# Skip if running locally with uv
import sys
if 'google.colab' in sys.modules:
    !pip install -q keras torch torchvision python-dotenv datasets transformers huggingface_hub
    print('Dependencies installed!')

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"

import keras
import torch
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import seaborn as sns

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## 2. Load Environment Variables

Load the HuggingFace token from a `.env` file (optional but needed for pushing to Hub).

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import os
hf_token = os.getenv("HF_TOKEN")

if hf_token:
    print("HuggingFace token loaded successfully.")
    from huggingface_hub import login
    login(token=hf_token)
else:
    print("No HF_TOKEN found in .env file.")
    print("You can still download public datasets and models.")
    print("Push to Hub will not be available without a token.")

## 3. Load the Beans Dataset

The Beans dataset contains images of bean leaves with 3 classes: angular leaf spot, bean rust, and healthy. It is a small, free dataset ideal for demonstrating fine-tuning.

In [ ]:
# Load the beans dataset from HuggingFace Hub
dataset = load_dataset("beans")

print(f"Dataset structure: {dataset}")
print(f"\nTraining samples: {len(dataset['train'])}")
print(f"Validation samples: {len(dataset['validation'])}")
print(f"Test samples: {len(dataset['test'])}")
print(f"\nFeatures: {dataset['train'].features}")
print(f"\nClass labels: {dataset['train'].features['labels'].names}")

In [ ]:
# Get class names
class_names = dataset["train"].features["labels"].names
num_classes = len(class_names)
print(f"Classes: {class_names}")

# Visualize some samples
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, class_name in enumerate(class_names):
    # Find first sample of each class
    for sample in dataset["train"]:
        if sample["labels"] == i:
            axes[i].imshow(sample["image"])
            axes[i].set_title(f"{class_name}", fontsize=14)
            axes[i].axis("off")
            break
plt.suptitle("Beans Dataset: One Sample Per Class", fontsize=16)
plt.tight_layout()
plt.show()

## 4. Prepare the Image Processor and Preprocessing

We use `AutoImageProcessor` to automatically apply the correct preprocessing for the ViT model.

In [ ]:
# Load the image processor for ViT
model_name = "google/vit-base-patch16-224"
image_processor = AutoImageProcessor.from_pretrained(model_name)

print(f"Image processor: {type(image_processor).__name__}")
print(f"Image size: {image_processor.size}")
print(f"Image mean: {image_processor.image_mean}")
print(f"Image std: {image_processor.image_std}")

In [ ]:
def preprocess(examples):
    """Preprocess images using the ViT image processor."""
    images = [img.convert("RGB") for img in examples["image"]]
    inputs = image_processor(images=images, return_tensors="pt")
    inputs["labels"] = examples["labels"]
    return inputs

# Apply preprocessing to all splits
processed_dataset = dataset.with_transform(preprocess)

# Verify preprocessing
sample = processed_dataset["train"][0]
print(f"Pixel values shape: {sample['pixel_values'].shape}")
print(f"Label: {sample['labels']}")

## 5. Load the Pre-trained ViT Model

We load a ViT model pre-trained on ImageNet and adapt it for our 3-class classification task.

In [ ]:
# Create label mappings
id2label = {i: label for i, label in enumerate(class_names)}
label2id = {label: i for i, label in enumerate(class_names)}

# Load the pre-trained ViT model for image classification
model = AutoModelForImageClassification.from_pretrained(
    model_name,
    num_labels=num_classes,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True  # Classification head size differs
)

print(f"Model: {model_name}")
print(f"Number of parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Number of trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"Output labels: {model.config.id2label}")

## 6. Configure Training Arguments

In [ ]:
# Define training arguments
training_args = TrainingArguments(
    output_dir="./vit-beans-finetuned",
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    remove_unused_columns=False,
    fp16=torch.cuda.is_available(),  # Use mixed precision on GPU
    report_to="none",  # Disable wandb/tensorboard logging
)

print("Training Arguments:")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Batch size (train): {training_args.per_device_train_batch_size}")
print(f"  Batch size (eval): {training_args.per_device_eval_batch_size}")
print(f"  Eval strategy: {training_args.eval_strategy}")
print(f"  FP16: {training_args.fp16}")

In [ ]:
def compute_metrics(eval_pred):
    """Compute accuracy for evaluation."""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc}

def collate_fn(batch):
    """Custom collate function for the dataloader."""
    return {
        "pixel_values": torch.stack([x["pixel_values"] for x in batch]),
        "labels": torch.tensor([x["labels"] for x in batch])
    }

## 7. Fine-Tune with the Trainer API

In [ ]:
# Create the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=processed_dataset["train"],
    eval_dataset=processed_dataset["validation"],
    compute_metrics=compute_metrics,
    data_collator=collate_fn,
)

print("Starting fine-tuning...")
print("=" * 50)

In [ ]:
# Train the model
train_result = trainer.train()

# Print training metrics
print("\nTraining completed!")
print(f"Training loss: {train_result.metrics['train_loss']:.4f}")
print(f"Training runtime: {train_result.metrics['train_runtime']:.1f}s")

## 8. Evaluate the Fine-Tuned Model

In [ ]:
# Evaluate on test set
test_results = trainer.evaluate(processed_dataset["test"])
print(f"Test Loss: {test_results['eval_loss']:.4f}")
print(f"Test Accuracy: {test_results['eval_accuracy']:.4f}")

In [ ]:
# Get predictions for detailed evaluation
predictions = trainer.predict(processed_dataset["test"])
y_pred = np.argmax(predictions.predictions, axis=-1)
y_true = predictions.label_ids

# Classification report
print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.title("Confusion Matrix - Fine-Tuned ViT on Beans", fontsize=14)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()

## 9. Visualize Predictions

In [ ]:
# Visualize some test predictions
test_dataset = dataset["test"]
probs = torch.nn.functional.softmax(torch.tensor(predictions.predictions), dim=-1).numpy()

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
np.random.seed(42)
indices = np.random.choice(len(test_dataset), 8, replace=False)

for i, ax in enumerate(axes.flat):
    idx = indices[i]
    sample = test_dataset[idx]
    ax.imshow(sample["image"])
    pred_label = class_names[y_pred[idx]]
    true_label = class_names[y_true[idx]]
    confidence = probs[idx][y_pred[idx]]
    color = "green" if pred_label == true_label else "red"
    ax.set_title(f"Pred: {pred_label} ({confidence:.2f})\nTrue: {true_label}",
                 color=color, fontsize=10)
    ax.axis("off")

plt.suptitle("Fine-Tuned ViT Predictions on Beans Test Set", fontsize=14)
plt.tight_layout()
plt.show()

## 10. (Optional) Push Model to HuggingFace Hub

If you have a HuggingFace token configured, you can push the fine-tuned model to the Hub.

In [ ]:
# Uncomment the following lines to push to the HuggingFace Hub
# (requires HF_TOKEN to be set in your .env file)

# if hf_token:
#     hub_model_name = "your-username/vit-beans-finetuned"
#     trainer.push_to_hub(hub_model_name)
#     image_processor.push_to_hub(hub_model_name)
#     print(f"Model pushed to: https://huggingface.co/{hub_model_name}")
# else:
#     print("Skipping push to Hub - no HF_TOKEN available.")

print("To push to Hub, uncomment the code above and set your HF_TOKEN.")
print("You can also save locally:")
trainer.save_model("./vit-beans-finetuned-final")
image_processor.save_pretrained("./vit-beans-finetuned-final")
print("Model saved locally to ./vit-beans-finetuned-final")